In [ ]:
#   Generate all 3×3 adjacency matrices with entries in [-1, 0, 1], -1=inhibition, 0=no interaction, 1=activation
#   Remove duplicates isomorphic networks
#   Save results using JLD2 in table-like format

using Combinatorics   #permutations
using JLD2            #saving to .jld2 file

#Define possible connection types 
const CONNECTION_TYPES = [-1, 0, 1]

# Generate all possible 3×3 networks 
function generate_limited_networks(max_networks_to_generate::Int)
    networks = Matrix{Int}[]
    count = 0
    for c11 in CONNECTION_TYPES, c12 in CONNECTION_TYPES, c13 in CONNECTION_TYPES,
        c21 in CONNECTION_TYPES, c22 in CONNECTION_TYPES, c23 in CONNECTION_TYPES,
        c31 in CONNECTION_TYPES, c32 in CONNECTION_TYPES, c33 in CONNECTION_TYPES

        adj_matrix = [c11 c21 c31;
                      c12 c22 c32;
                      c13 c23 c33]

        push!(networks, adj_matrix)
        count += 1

        if count >= max_networks_to_generate
            break
        end
    end
    return networks
end

# Canonicalize a network
function canonicalize_network(adj_matrix::AbstractMatrix{Int})
    n = size(adj_matrix, 1)
    min_repr = adj_matrix
    min_hash = hash(adj_matrix)

    for p in permutations(1:n)
        perm_matrix = zeros(Int, n, n)
        for i in 1:n
            for j in 1:n
                perm_matrix[p[i], p[j]] = adj_matrix[i, j]
            end
        end

        current_hash = hash(perm_matrix)
        if current_hash < min_hash
            min_hash = current_hash
            min_repr = perm_matrix
        end
    end

    return min_repr
end

# Get unique networks up to permutation 
function get_unique_networks(all_networks)
    unique_network_set = Set{Matrix{Int}}()
    for network in all_networks
        canonical = canonicalize_network(network)
        push!(unique_network_set, canonical)
    end
    return collect(unique_network_set)
end

# Master function: generate, deduplicate, assign IDs, save in table-like format
function generate_and_save_unique_networks(max_networks_to_generate::Int, filename::String)
    println("=================================================")
    println("Generating networks...")
    println("=================================================")
    all_networks = generate_limited_networks(max_networks_to_generate)
    total_generated = length(all_networks)
    println("Total networks generated (including duplicates): $total_generated")

    println("\nFinding unique networks (removing isomorphic ones)...")
    unique_networks = get_unique_networks(all_networks)
    num_unique = length(unique_networks)

    println("-------------------------------------------------")
    println("Total unique networks found: $num_unique")
    println("Duplicates removed: $(total_generated - num_unique)")
    println("-------------------------------------------------")

    #Create table-like vector of NamedTuples 
    network_table = [ (unique_id = i, adj_matrix = net) for (i, net) in enumerate(unique_networks) ]

    println("\nSaving unique networks with table format to '$filename'...")
    @save filename network_table num_unique total_generated
    println("Unique networks saved successfully!")

    return network_table, num_unique
end

# --- Run the process ---
# Use max_networks_to_generate = 3^9 = 19683 for all possible 3x3 networks
println("\n--- Running unique network generation ---\n")
network_table, num_unique = generate_and_save_unique_networks(19683, "unique_networks_table.jld2")

println(" Done!")
println("Total unique networks saved: $num_unique")
println("File: unique_networks_table.jld2")